# Torch Model

This notebook is simply a reimplementation of the [previous notebook](01_cse_selection.ipynb) using Torch instead of TensorFlow.

We also use the `optuna` library to find optimial model parameters instead of doing so ourself.

In [1]:
# --------------------------------------------------------------------------------------------
# Constants and parameters

# Update these values to point at your local build
MCH_FILE = "~/git/runtime/artifacts/spmi/mch/43854594-cd60-45df-a89f-5b7697586f46.linux.x64/libraries_tests_no_tiered_compilation.run.linux.x64.Release.mch"
CORE_ROOT = "~/git/runtime/artifacts/bin/coreclr/linux.x64.Checked/"

# At what perf_score would we want to select a CSE?  We don't want to select any CSE which
# is >0.0, as we are creating new temporaries for no value.
# I've arbitrarily selected this minimum perf_score improvement for a "successful" CSE.
CSE_SUCCESS_THRESHOLD = -5.0

# We will only retain features which have a correlation of at least 1% with the change in
# perf_score.  This is to reduce the number of features we have to consider.  Selecting a
# value as high as 15% would still be reasonable, but since there are so few features we
# can afford to allow a lower threshold.
CORRELATION_THRESHOLD = 0.01

# The reinforcement learning trained model to compare against.
RL_MODEL = "../models/rl/ppo.zip"
MODEL_ALGORITHM = 'PPO'

# Single CSE Classification Model parameters
OPTIMIZER = 'adam'
LOSS = 'binary_crossentropy'
METRICS = ['accuracy']
EPOCHS = 100
BATCH_SIZE = 256

# --------------------------------------------------------------------------------------------
# Setup and imports

# resolve '~' to home directory
import os
MCH_FILE = os.path.expanduser(MCH_FILE)
CORE_ROOT = os.path.expanduser(CORE_ROOT)

import os
import sys
from tqdm import tqdm
from pandas import DataFrame
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Add the parent directory to the path so that we can import the jitml module
sys.path.append(os.path.dirname(os.getcwd()))

from jitml import SuperPmiContext, MethodKind, SuperPmiCache, SuperPmi, get_individual_cse_perf, JitType

ctx = SuperPmiContext(mch=MCH_FILE, core_root=CORE_ROOT)
cache : SuperPmiCache = ctx.create_cache()
spmi : SuperPmi = ctx.create_superpmi()
spmi.start()

def calculate_geomean(scores, baselines):
    ratios = [score / baseline for score, baseline in zip(scores, baselines)]
    log_ratios = np.log(ratios)
    mean_log_ratios = np.mean(log_ratios)
    geomean = np.exp(mean_log_ratios)
    return geomean

def print_difference(scores, baseline, name : str, baseline_name : str):
    if not scores or not baseline:
        print(f"No data for {name} or {baseline_name}")
        return

    same_as_no_cse = 0
    bad_change = []
    good_change = []

    for i in range(len(scores)):
        if scores[i] is None:
            continue

        diff = scores[i] - baseline[i]
        if np.isclose(diff, 0):
            same_as_no_cse += 1
        elif diff < 0:
            good_change.append(diff / scores[i] * 100.0)
        else:
            bad_change.append(diff / scores[i] * 100.0)

    print(f"Geomean of {name} vs {baseline_name}: {calculate_geomean(scores, baseline):.2f}")
    print()
    print(f"% of time same score:          {100 * same_as_no_cse / len(scores):.2f}%")
    print(f"% of time {baseline_name} is better:    {100 * len(bad_change) / len(scores):.2f}%")
    print(f"% of time {name} is better: {100 * len(good_change) / len(scores):.2f}%")
    print()
    print(f"Average improvement when {name} is better: {np.mean(good_change):.2f}%")
    print(f"Average degradation when {baseline_name} is better:     {np.mean(bad_change):.2f}%")
    print()


2024-08-29 12:43:47.650400: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-29 12:43:48.092740: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-29 12:43:48.092919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-29 12:43:48.153041: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-29 12:43:48.288996: I tensorflow/core/platform/cpu_feature_guar

## Model Definition

Define our `[64, 64, 64, 1]` model in Torch instead of TensorFlow.

In [2]:
class CSEModel(nn.Module):
    def __init__(self, input_len):
        super().__init__()
        self.layer1 = nn.Linear(input_len, 64)
        self.layer2 = nn.Linear(64, 64)
        self.layer3 = nn.Linear(64, 64)
        self.output = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = self.sigmoid(self.output(x))
        return x

def split_and_scale(df):
    train_mask = df['method'].isin(cache.train_methods)
    test_mask = df['method'].isin(cache.test_methods)
    x_train, y_train = df[train_mask].drop(columns=['target', 'method']).values, df[train_mask]['target'].values
    x_test, y_test = df[test_mask].drop(columns=['target', 'method']).values, df[test_mask]['target'].values

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    feature_len = x_train.shape[1]

    x_train_torch = torch.tensor(x_train, dtype=torch.float32).to(DEVICE)
    x_test_torch = torch.tensor(x_test, dtype=torch.float32).to(DEVICE)
    y_train_torch = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    y_test_torch = torch.tensor(y_test, dtype=torch.float32).to(DEVICE)
    return scaler, x_train_torch, x_test_torch, y_train_torch, y_test_torch, feature_len

def sanitize_data(df, threshold):
    result = df.copy()

    for member in JitType:
        result[f"type_{member.name}"] = result['type'] == member

    result.drop(columns=['type'], inplace=True)
    result['selected'] = result['selected'].apply(len)
    result['target'] = result['diff'] < threshold

    result = result[result['viable']]

    to_drop = ['cse_index', 'cse_score', 'no_cse_score', 'heuristic_score', 'heuristic_selected',
               'index', 'applied', 'viable', 'diff']
    to_drop = [x for x in to_drop if x in result.columns]

    result.drop(columns=to_drop, inplace=True)
    return result

def train_single_cse_model(data, threshold):
    normalized = sanitize_data(data, threshold)
    scaler, x_train, x_test, y_train, y_test, feature_len = split_and_scale(normalized)

    print(f"Training on {len(x_train)} CSE decisions.")
    print(f"Validating on {len(x_test)} CSE decisions.")

    model = CSEModel(feature_len).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_dataset = TensorDataset(x_train, y_train)
    test_dataset = TensorDataset(x_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    epoch_pbar = tqdm(range(EPOCHS), desc='Training Epochs', ncols=120)

    for epoch in epoch_pbar:
        model.train()
        train_loss = 0
        correct_train = 0
        total_train = 0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct_train += (predicted == targets).sum().item()
            total_train += targets.size(0)

        train_accuracy = correct_train / total_train

        model.eval()
        test_loss = 0
        correct_test = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                outputs = model(inputs).squeeze()
                loss = criterion(outputs, targets)
                test_loss += loss.item()
                predicted = (outputs > 0.5).float()
                correct_test += (predicted == targets).sum().item()

        test_accuracy = correct_test / len(test_dataset)

        train_loss /= len(train_loader)
        test_loss /= len(test_loader)

        epoch_pbar.set_postfix({
            'train_acc': train_accuracy,
            'test_acc': test_accuracy
        })

        #print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


    return scaler, model

individual_cse = get_individual_cse_perf(MCH_FILE, CORE_ROOT)
single_scalar, single_model = train_single_cse_model(individual_cse, CSE_SUCCESS_THRESHOLD)


Training on 291304 CSE decisions.
Validating on 32401 CSE decisions.


/home/leculver/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training Epochs:  59%|██████████████████▉             | 59/100 [03:48<02:32,  3.72s/it, train_acc=0.975, test_acc=0.973]

## Validate the Model

We get the same result as TensorFlow, as expected.

In [4]:
SELECTION_PROBABILITY = 0.5

def predict_cse_with_model(scalar, model, threshold):
    model_scores = []
    no_cse_scores = []
    heuristic_scores = []
    chosen = []
    chosen_pct = []

    grouped = individual_cse.groupby('method')
    for method_id in tqdm(cache.test_methods, ncols=120, desc="Predicting CSEs"):
        # Sanitize data
        if method_id not in grouped.groups:
            continue

        row = grouped.get_group(method_id)
        row_viable = row[row['viable']]
        features = sanitize_data(row_viable, threshold)
        x = scalar.transform(features.drop(columns=['target', 'method']).values)

        # Convert features to PyTorch tensor
        x_tensor = torch.tensor(x, dtype=torch.float32).to(DEVICE)

        # Predict what CSEs to use
        model.eval()
        with torch.no_grad():
            y_pred = model(x_tensor).cpu().squeeze().numpy()

        above_threshold = np.where(y_pred > SELECTION_PROBABILITY)[0]
        cses_chosen = above_threshold[np.argsort(-y_pred[above_threshold])].tolist()

        # pull index out of row_viable
        cses_chosen = [row_viable.iloc[x].cse_index for x in cses_chosen]

        # JIT the method with the chosen CSEs
        no_cse = cache.jit_method(spmi, method_id, MethodKind.NO_CSE)
        heuristic = cache.jit_method(spmi, method_id, MethodKind.HEURISTIC)

        if cses_chosen:
            method = cache.jit_method(spmi, method_id, cses_chosen)
            if method is None or np.isclose(method.perf_score, 0.0):
                continue
        else:
            method = no_cse

        model_scores.append(method.perf_score)
        no_cse_scores.append(no_cse.perf_score)
        heuristic_scores.append(heuristic.perf_score)
        chosen.append(len(cses_chosen))
        chosen_pct.append(len(cses_chosen) / sum(1 for x in no_cse.cse_candidates if x.viable))

    print(f"Average CSEs chosen:   {np.mean(chosen):.2f}")
    print(f"Average CSEs chosen %: {np.mean(chosen_pct) * 100:.2f}%")
    print()
    print("VS No CSE")
    print_difference(model_scores, no_cse_scores, "model", "no-cse")
    print()
    print("VS Heuristic")
    print_difference(model_scores, heuristic_scores, "model", "heuristic")

print(f"Results from selecting CSEs with predicted perfscore < {CSE_SUCCESS_THRESHOLD}:")
predict_cse_with_model(single_scalar, single_model, CSE_SUCCESS_THRESHOLD)

NEW_THRESHOLD = 0.0
scalar_zero, model_zero = train_single_cse_model(individual_cse, NEW_THRESHOLD)

print()
print(f"Results from selecting CSEs with predicted perfscore < {NEW_THRESHOLD}:")
predict_cse_with_model(scalar_zero, model_zero, NEW_THRESHOLD)


Results from selecting CSEs with predicted perfscore < -5.0:


Predicting CSEs: 100%|█████████████████████████████████████████████████████████████| 6140/6140 [00:50<00:00, 121.92it/s]


Average CSEs chosen:   0.50
Average CSEs chosen %: 8.99%

VS No CSE
Geomean of model vs no-cse: 0.98

% of time same score:          76.82%
% of time no-cse is better:    0.88%
% of time model is better: 22.30%

Average improvement when model is better: -10.98%
Average degradation when no-cse is better:     1.43%


VS Heuristic
Geomean of model vs heuristic: 1.00

% of time same score:          17.27%
% of time heuristic is better:    48.67%
% of time model is better: 34.06%

Average improvement when model is better: -5.72%
Average degradation when heuristic is better:     3.30%



## Find Optimal Model Hyperparameters

Use `optuna` to find optimal model parameters.

In [18]:
import torch.optim as optim
import optuna
from sklearn.metrics import precision_score

class CSEModelParameterized(nn.Module):
    def __init__(self, input_len, n_layers, n_units, dropout):
        super().__init__()
        layers = []
        for i in range(n_layers):
            if i == 0:
                layers.append(nn.Linear(input_len, n_units))
            else:
                layers.append(nn.Linear(n_units, n_units))

            layers.append(nn.ReLU())
            if not np.isclose(dropout, 0.0):
                layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(n_units, 1))
        layers.append(nn.Sigmoid())

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

best_model_state = None
best_precision = 0
feature_len = 0
new_scalar = None

def objective(trial):
    global best_model_state, best_precision, feature_len, new_scalar

    normalized = sanitize_data(individual_cse, NEW_THRESHOLD)
    new_scalar, x_train, x_test, y_train, y_test, feature_len = split_and_scale(normalized)

    epochs = trial.suggest_int('epochs', 100, 250, step=50)
    n_layers = trial.suggest_int('n_layers', 1, 4)
    n_units = trial.suggest_int('n_units', 32, 128, step=32)
    dropout = trial.suggest_float('dropout', 0.0, 0.5, step=0.1)
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)

    model = CSEModelParameterized(feature_len, n_layers, n_units, dropout).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(x_train).squeeze()
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        output = model(x_test).squeeze()
        pred = (output >= 0.5).float()

    precision = precision_score(y_test.cpu(), pred.cpu())

    if precision > best_precision:
        best_precision = precision
        best_model_state = model.state_dict()

    return precision

def find_best_model():
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=250, show_progress_bar=True)

    print(f"Best trial: {study.best_trial}")
    print(f"Best parameters: {study.best_params}")
    print(f"Best precision: {study.best_value}")

    best_model = CSEModelParameterized(feature_len, study.best_params['n_layers'], study.best_params['n_units'], study.best_params['dropout']).to(DEVICE)
    best_model.load_state_dict(best_model_state)

    return best_model

best_model = find_best_model()

Best trial: 79. Best value: 0.930656: 100%|██████████| 250/250 [09:16<00:00,  2.22s/it]

Best trial: FrozenTrial(number=79, state=TrialState.COMPLETE, values=[0.9306563039723661], datetime_start=datetime.datetime(2024, 6, 4, 10, 31, 11, 81108), datetime_complete=datetime.datetime(2024, 6, 4, 10, 31, 14, 301941), params={'epochs': 200, 'n_layers': 4, 'n_units': 96, 'dropout': 0.2, 'lr': 0.05788391826300213}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'epochs': IntDistribution(high=250, log=False, low=100, step=50), 'n_layers': IntDistribution(high=4, log=False, low=1, step=1), 'n_units': IntDistribution(high=128, log=False, low=32, step=32), 'dropout': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'lr': FloatDistribution(high=0.1, log=True, low=0.0001, step=None)}, trial_id=79, value=None)
Best parameters: {'epochs': 200, 'n_layers': 4, 'n_units': 96, 'dropout': 0.2, 'lr': 0.05788391826300213}
Best precision: 0.9306563039723661


## Validate New Model

Validate that the new model performs as well or better than the previous version.  It turns out to have similar performance.

In [19]:
print(f"Results from best model selecting CSEs with predicted perfscore < {NEW_THRESHOLD}:")
predict_cse_with_model(new_scalar, best_model, NEW_THRESHOLD)

Results from best model selecting CSEs with predicted perfscore < 0.0:


Predicting CSEs: 100%|██████████████████████████████████████████████████████████████| 6140/6140 [01:42<00:00, 59.91it/s]


Average CSEs chosen:   1.89
Average CSEs chosen %: 37.34%

VS No CSE
Geomean of model vs no-cse: 0.96

% of time same score:          29.99%
% of time no-cse is better:    1.92%
% of time model is better: 68.09%

Average improvement when model is better: -6.66%
Average degradation when no-cse is better:     1.44%


VS Heuristic
Geomean of model vs heuristic: 0.98

% of time same score:          27.20%
% of time heuristic is better:    10.72%
% of time model is better: 62.08%

Average improvement when model is better: -4.00%
Average degradation when heuristic is better:     2.06%

